In [1]:
data = "../submissions/sample_submission.tsv"

In [11]:
import pandas as pd
import csv

In [12]:
# Read with csv module first for maximum control
all_rows = []
max_fields = 0

with open(data, 'r', encoding='utf-8') as f:
    reader = csv.reader(f, delimiter='\t', quoting=csv.QUOTE_MINIMAL)
    for row in reader:
        all_rows.append(row)
        max_fields = max(max_fields, len(row))

# Ensure all rows have same length by padding with None
for i, row in enumerate(all_rows):
    if len(row) < max_fields:
        all_rows[i] = row + [None] * (max_fields - len(row))

# Convert to DataFrame
df = pd.DataFrame(all_rows)

# If first row looks like headers, use it
if df.shape[0] > 1:
    # Check if first row looks like header (non-numeric, unique values)
    first_row = df.iloc[0]
    if first_row.apply(lambda x: isinstance(x, str) and not x.replace('.', '').isdigit()).all():
        df.columns = first_row
        df = df[1:].reset_index(drop=True)

print(f"Successfully read {len(df)} rows with {len(df.columns)} columns")
print(df.head())

Successfully read 20004 rows with 4 columns
            0           1      2  \
0  A0A0C5B5G6  GO:0000001  0.123   
1  A0A0C5B5G6  GO:0000002  0.456   
2  A0A0C5B5G6        Text  0.123   
3  A0A0C5B5G6        Text  0.456   
4  A0A0C5B5G6        Text  0.456   

                                                   3  
0                                               None  
1                                               None  
2  Regulates insulin sensitivity and metabolic ho...  
3  Inhibits the folate cycle, thereby reducing de...  
4  and the activation of the metabolic regulator ...  


In [13]:
# This one line replaces your entire script
df = pd.read_csv(
    data, 
    sep='\t', 
    header='infer', 
    on_bad_lines='warn', # Handles the "padding" issue transparently
    engine='c'           # Maximum speed/memory efficiency
)

print(f"Successfully read {len(df)} rows with {len(df.columns)} columns")
print(df.head())

Successfully read 19999 rows with 3 columns
   A0A0C5B5G6  GO:0000001  0.123
0  A0A0C5B5G6  GO:0000002  0.456
1  A0A1B0GTW7  GO:0000001  0.123
2  A0A1B0GTW7  GO:0000002  0.456
3      A0JNW5  GO:0000001  0.123
4      A0JNW5  GO:0000002  0.456


/tmp/ipykernel_31401/3143984755.py:2: ParserWarning: Skipping line 3: expected 3 fields, saw 4
Skipping line 4: expected 3 fields, saw 4
Skipping line 5: expected 3 fields, saw 4
Skipping line 8: expected 3 fields, saw 4

  df = pd.read_csv(


In [14]:
df = pd.read_csv(
    data, 
    sep='\t', 
    header=None,               # Don't let it steal your first row!
    names=['ID', 'GO', 'Val'], # Give them proper names
    index_col=False,           # Forces pandas not to use the first col as index
    on_bad_lines='warn'        # Keep this to see where the file is "dirty"
)

df.head()

/tmp/ipykernel_31401/1019853811.py:1: ParserWarning: Skipping line 3: expected 3 fields, saw 4
Skipping line 4: expected 3 fields, saw 4
Skipping line 5: expected 3 fields, saw 4
Skipping line 8: expected 3 fields, saw 4

  df = pd.read_csv(


,ID,GO,Val
0,A0A0C5B5G6,GO:0000001,0.123
1,A0A0C5B5G6,GO:0000002,0.456
2,A0A1B0GTW7,GO:0000001,0.123
3,A0A1B0GTW7,GO:0000002,0.456
4,A0JNW5,GO:0000001,0.123


In [15]:
# We define 4 names because the parser found at least 4 columns
col_names = ['ID', 'GO', 'Val', 'Extra']

df = pd.read_csv(
    data, 
    sep='\t', 
    header=None, 
    names=col_names,
    engine='python' # The Python engine is more flexible with varying line lengths
)

print(f"New shape: {df.shape}")
print(df.iloc[[2, 3, 4]]) # Look at the lines that were previously skipped

New shape: (20004, 4)
           ID    GO    Val                                              Extra
2  A0A0C5B5G6  Text  0.123  Regulates insulin sensitivity and metabolic ho...
3  A0A0C5B5G6  Text  0.456  Inhibits the folate cycle, thereby reducing de...
4  A0A0C5B5G6  Text  0.456  and the activation of the metabolic regulator ...


In [24]:
import pandas as pd
import re

rows = []
current_row = None

# Pattern: Starts with a letter/number, not a space or "and the activation..."
# Adjust the regex based on your specific ID format
id_pattern = re.compile(r'^[A-Z0-9]{6,}') 

with open(data, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip('\n').split('\t')
        
        # Check if the first part looks like a Protein ID
        if id_pattern.match(parts[0]):
            if current_row:
                rows.append(current_row)
            current_row = parts
        else:
            # It's a "broken" line! Glue it to the last field of the previous row
            if current_row:
                current_row[-1] += " " + " ".join(parts)

# Add the final row
if current_row:
    rows.append(current_row)

# Create DataFrame (padding only if some rows actually had fewer than 4 columns)
df = pd.DataFrame(rows, columns=['ID', 'GO', 'Val', 'Extra'])

df['Extra'][2]

'Regulates insulin sensitivity and metabolic homeostasis'

In [18]:
import pandas as pd

# 1. Read everything as strings initially to prevent data loss
df_raw = pd.read_csv(
    data, 
    sep='\t', 
    header=None, 
    names=['Col1', 'Col2', 'Col3', 'Col4'], 
    on_bad_lines='warn',
    engine='python'
)

# 2. Repair the logic: If Col2 is "Text", the real GO term might be missing 
# or shifted. We need to align the 'Val' correctly.
def repair_cafa_row(row):
    # If Col2 is "Text", our Score (Val) is actually in Col3
    if row['Col2'] == 'Text':
        return pd.Series([row['Col1'], 'UNKNOWN_OR_MISSING', row['Col3']])
    else:
        # Standard row: ID, GO, Score
        return pd.Series([row['Col1'], row['Col2'], row['Col3']])

# Apply repair to get a clean 3-column structure
df_clean = df_raw.apply(repair_cafa_row, axis=1)
df_clean.columns = ['ProteinID', 'GO_Term', 'Score']

# 3. Convert Score to float for CAFA validation
df_clean['Score'] = pd.to_numeric(df_clean['Score'], errors='coerce')

print(df_clean.head())

    ProteinID             GO_Term  Score
0  A0A0C5B5G6          GO:0000001  0.123
1  A0A0C5B5G6          GO:0000002  0.456
2  A0A0C5B5G6  UNKNOWN_OR_MISSING  0.123
3  A0A0C5B5G6  UNKNOWN_OR_MISSING  0.456
4  A0A0C5B5G6  UNKNOWN_OR_MISSING  0.456


In [27]:
# 1. Ensure Score is a number
df_clean['Score'] = pd.to_numeric(df_clean['Score'], errors='coerce')

# 2. Define what a 'Valid' GO Term looks like (must start with 'GO:')
is_valid_go = df_clean['GO_Term'].str.startswith('GO:', na=False)

# 3. Filter for quality
final_submission = df_clean[is_valid_go].dropna(subset=['Score'])

# 4. Remove duplicates (multilabel data shouldn't have the same GO for same Protein twice)
final_submission = final_submission.drop_duplicates(subset=['ProteinID', 'GO_Term'])

print(f"Original messy rows: {len(df_raw)}")
print(f"Valid CAFA rows: {len(final_submission)}")
print(final_submission.head(20))

Original messy rows: 20004
Valid CAFA rows: 20000
     ProteinID     GO_Term  Score
0   A0A0C5B5G6  GO:0000001  0.123
1   A0A0C5B5G6  GO:0000002  0.456
5   A0A1B0GTW7  GO:0000001  0.123
6   A0A1B0GTW7  GO:0000002  0.456
8       A0JNW5  GO:0000001  0.123
9       A0JNW5  GO:0000002  0.456
10      A0JP26  GO:0000001  0.123
11      A0JP26  GO:0000002  0.456
12      A0PK11  GO:0000001  0.123
13      A0PK11  GO:0000002  0.456
14      A1A4S6  GO:0000001  0.123
15      A1A4S6  GO:0000002  0.456
16      A1A519  GO:0000001  0.123
17      A1A519  GO:0000002  0.456
18      A1L190  GO:0000001  0.123
19      A1L190  GO:0000002  0.456
20      A1L3X0  GO:0000001  0.123
21      A1L3X0  GO:0000002  0.456
22      A1X283  GO:0000001  0.123
23      A1X283  GO:0000002  0.456
